# Quickstart: compile, simulate, target, search

The command-line walkthrough, run here on the two-block example in `examples/synthetic`.
Its numbers are checkable by hand: with no schedule the natural-track volume at year 0 is
`3.0 * 120 + 2.0 * 170 = 700`, and clearing block 1 leaves `2.0 * 170 = 340`.

## Compile a dataset

In [1]:
!ws4 build --xml examples/synthetic/recipe.xml --fragments examples/synthetic/fragments.csv --out tmp/examples/quickstart/compiled

Built dataset → tmp/examples/quickstart/compiled


In [2]:
!ws4 validate --dataset tmp/examples/quickstart/compiled

Dataset is valid → tmp/examples/quickstart/compiled


## Simulate a schedule

A schedule is a CSV of `block_id`, `treatment`, and `period`, where the period is in years.

In [3]:
from pathlib import Path

work = Path("tmp/examples/quickstart")
work.mkdir(parents=True, exist_ok=True)
(work / "schedule.csv").write_text("block_id,treatment,period\n1,CC,0\n")

!ws4 simulate --dataset tmp/examples/quickstart/compiled --schedule tmp/examples/quickstart/schedule.csv --years 5 --out tmp/examples/quickstart/sim

Simulation complete → tmp/examples/quickstart/sim


In [4]:
import pandas as pd

values = pd.read_csv(work / "sim" / "account_values.csv")
print(values.head(4).to_string(index=False))

 year        account  value
    0 feature.Volume  340.0
    0 product.Volume  360.0
    1 feature.Volume  360.0
    2 feature.Volume  380.0


`340.0` is the figure from the example: block 1's 3.0 ha of natural track is gone, and block
2's 2.0 ha of natural volume remains.

## State a target and check the run against it

In [5]:
(work / "targets.yaml").write_text(
    "targets:\n"
    "- account: feature.Volume\n"
    "  year: 0\n"
    "  minimum: 700\n"
    "  maximum: 700\n"
    "  period_width: 1\n"
)

!ws4 report --run tmp/examples/quickstart/sim --targets tmp/examples/quickstart/targets.yaml

Report written → tmp/examples/quickstart/sim


In [6]:
print(pd.read_csv(work / "sim" / "report_targets.csv").to_string(index=False))

       account  year  value  minimum  maximum  penalty status
feature.Volume     0  340.0      700      700    360.0  below


The status is `below`: clearing block 1 drops the natural volume to 340 against a floor of
700, and the penalty is the shortfall.

## Search for a schedule that meets it

In [7]:
(work / "weights.yaml").write_text("weights: {}\n")

!ws4 optimize --dataset tmp/examples/quickstart/compiled --targets tmp/examples/quickstart/targets.yaml --weights tmp/examples/quickstart/weights.yaml --years 5 --time-limit 00:00:05 --seed 1 --threads 1 --out tmp/examples/quickstart/opt

                              Optimization Summary                              
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ key                 ┃ value                                                  ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ incumbent_objective │ 0.0                                                    │
│ iterations          │ 500                                                    │
│ seed                │ 1                                                      │
│ metaheuristic       │ sa                                                     │
│ starts              │ 1                                                      │
│ blocks_scheduled    │ 0                                                      │
│ operators           │ ['toggle', 'shift', 'swap', 'lns', 'cluster', 'merge', │
│                     │ 'split']                                               │
│ status              │ ok  

In [8]:
import json

summary = json.loads((work / "opt" / "summary.json").read_text())
print({key: summary[key] for key in ("incumbent_objective", "blocks_scheduled", "iterations", "seed", "status")})
print((work / "opt" / "schedule.csv").read_text())

{'incumbent_objective': 0.0, 'blocks_scheduled': 0, 'iterations': 500, 'seed': 1, 'status': 'ok'}
block_id,treatment,period



The incumbent is an **empty schedule**: leaving the stand alone holds `feature.Volume` at 700,
which meets the floor, so nothing can beat a penalty of zero. On a two-block landscape that is
the whole answer.

## Watch a search

`ws4 watch` paints the realized-versus-target window; `--plain` prints one frame instead of the
live view, which is what a notebook wants.

In [9]:
!ws4 watch --dataset tmp/examples/quickstart/compiled --targets tmp/examples/quickstart/targets.yaml --account feature.Volume --years 5 --period-width 1 --iterations 200 --plain

╭───────────────────────────────── ws4 watch ──────────────────────────────────╮
│ feature.Volume   objective (sum of penalties): 0                             │
│ iteration 200/200   4643.40 eval/s   elapsed 0.0s   seed 1   sa              │
╰──────────────────────────────────────────────────────────────────────────────╯

feature.Volume   Z +0                                                           
    748.1 │  ::::::::                                                           
          │####::::::                                                           
          │####::::::                                                           
          │####::::::                                                           
          │####::::::                                                           
          │####::::::                                                           
          │####::::##                                                           
          │##########      

## Where to go next

* The how-to guides, for targets with teeth, even flow, and patch size distributions.
* `020_even_flow.ipynb`, which uses the Python API on a larger estate.